In [ ]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
folder=r'folder'
summaryFile='Scraping_List.txt'
st=pd.read_csv(folder+'\\'+summaryFile)
print(st)
#Define Product to extract
search_text = st['Product Name'][9]
print(search_text)

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:
            if 'Details' in f:
                if search_text in f:
                    file_list.append(f)
file_list

In [224]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [ ]:
file_link

In [ ]:
df=pd.DataFrame()
for file in file_link:
    df1=pd.read_excel(file)
    df1["SourceName"]=file
    #df['SourceName'] = df['SourceName'].str.replace('//', '')
    df1['SourceName'] = df1['SourceName'].str.rsplit('''\\''',1,expand=True)[1]
    df=df.append(df1)

In [227]:
df=df.reset_index()
del df['Sl.No']
del df['index']

In [ ]:
cols=df.columns.to_list()
remove=['Source',"SourceName"]
colsr=[ele for ele in cols if ele not in remove]
cols=remove+colsr
print(cols)
df = df[cols]

In [ ]:
df["SourceName"].value_counts()

In [230]:
dfAtt=df[['Links',"Name",'Attributes','Source']]
dfAtt=dfAtt.dropna()
dfAtt=dfAtt.reset_index()

In [231]:
from ast import literal_eval
for i in range(len(dfAtt)):
    dfAtt.at[i,'Attributes']=literal_eval(dfAtt['Attributes'][i])

In [ ]:
dfAtt=dfAtt.explode('Attributes')
dfAtt[['Attributes', 'Value']] = dfAtt['Attributes'].str.split(':',1, expand=True)
dfAtt

In [ ]:
cols=["Source",
"Name",
"Attributes",
"Value",
"Links"
]
dfAtt = dfAtt[cols]
dfAtt=dfAtt.drop_duplicates()
dfAtt.info()

In [ ]:
dfAtt

In [ ]:
summary_df = pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count',columns="Source").reset_index()
totalsummarydf=pd.pivot_table(dfAtt, values='Value', index='Attributes',aggfunc='count').reset_index()
totalsummarydf
summary_df=pd.merge(summary_df,totalsummarydf,on='Attributes')
summary_df
summary_df =summary_df.sort_values(by='Value',ascending=False)
summary_df=summary_df.rename(columns ={'Value':'No. of Products contains this Attribute'})
summary_df

In [236]:
df['Name_Clean']=df['Name'].str.lower().str.replace("compatible with","").str.replace('w/',"with ")
for i in range(len(df)):
    if "washer " in df['Name_Clean'][i] and "reservoir" in df['Name_Clean'][i]:

        if 'cap'in df['Name_Clean'][i] and 'with' in df['Name_Clean'][i]:
                df.loc[i,'Product']="Washer Fluid Reservoir With Cap"
        elif 'grommet'in df['Name_Clean'][i] and 'with' in df['Name_Clean'][i]:
            df.loc[i,'Product']="Washer Fluid Reservoir With Grommet"
        elif 'hose'in df['Name_Clean'][i] and 'with' in df['Name_Clean'][i]:
            df.loc[i,'Product']="Washer Fluid Reservoir With Hose"
        elif 'cap 'in df['Name_Clean'][i] and not 'with' in df['Name_Clean'][i]:
            df.loc[i,'Product']="Cap"
        elif 'grommet' in df['Name_Clean'][i]:
            df.loc[i,"Product"]='Grommet'
        elif 'hose'in df['Name_Clean'][i] and not 'with' in df['Name_Clean'][i]:
            df.loc[i,'Product']="Hose"
        elif "washer " in df['Name_Clean'][i] and "reservoir" in df['Name_Clean'][i]:
            df.loc[i,'Product']="Washer Fluid Reservoir"
    elif'pump'in df['Name_Clean'][i]:
        df.loc[i,"Product"]="Washer Pump"
    else:
        df.loc[i,"Product"]="Other Product"

del df['Name_Clean']

In [ ]:
df['Product'].value_counts()

In [ ]:
OFolder=r"Scraped_Files"

In [239]:
with pd.ExcelWriter(OFolder+'\\'+f'Consol_Scrapped_Data_'+search_text+'.xlsx') as writer:  # doctest: +SKIP
    df.to_excel(writer,index=False, sheet_name='Raw')
    summary_df.to_excel(writer,index=False, sheet_name='Attribute_Summary')
    dfAtt.to_excel(writer,index=False, sheet_name='Attribute_Value')

In [ ]:
df.info()

In [ ]:
print(OFolder+'\\'+f'Consol_Scrapped_Data_'+search_text+'.xlsx')